# Appendix L — Laboratory

**Book:** *Brain–Computer Interfaces and Cognitive Computing: An Introduction*  
**Prerequisites:** Chapters 0–12.  
**Author:** Er. Rishabh Aryan

Four sessions. No purchased headset is required. Every numeric cell runs on synthetic traces that obey the same bookkeeping as EEG. Optional notes point to public datasets (MOABB, PhysioNet EEG Motor Movement/Imagery, BNCI Horizon) if the instructor has downloads enabled.

**Rules that do not relax**

- Causal splits. No shuffled overlapping windows as a test set.  
- Chance level and a linear baseline beside every score.  
- No medical claim about a classmate’s brain.  
- Public-data licences apply; no re-identification games (Chapter 12).  
- Fill the reporting bundle (Definition 12.1) before you call a number a result.

**Software.** NumPy and Matplotlib, as in the rest of the book. MNE-Python is optional and never assumed.


---

## L0 Shared generators

Run this cell first in every session. It defines small simulators so that later cells stay short.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(12)
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})


def simulate_eeg(C=8, seconds=8.0, fs=256, line=50.0):
    T = int(seconds * fs)
    t = np.arange(T) / fs
    X = 0.4 * np.random.randn(C, T)
    for c in range(C):
        X[c] += 0.25 * np.sin(2 * np.pi * (9 + 0.3 * c) * t)
        X[c] += 0.15 * np.sin(2 * np.pi * line * t)
    return t, X, fs


def simulate_imagery_blocks(n_blocks=8, n_per=30, C=8, fs=256, win=1.0, drift=0.12):
    '''Feature-space imagery: mu-power-like z with block drift. Labels 0/1.'''
    d = C
    Z, y, block = [], [], []
    for b in range(n_blocks):
        shift = drift * b
        for _ in range(n_per):
            lab = np.random.randint(0, 2)
            z = np.random.randn(d) * 0.8
            if lab == 1:
                z[:3] += 1.0 - shift * 0.3
            else:
                z[3:6] += 1.0 + shift * 0.2
            Z.append(z)
            y.append(lab)
            block.append(b)
    return np.array(Z), np.array(y), np.array(block)


def simulate_p300(n_rare=12, n_freq=60, fs=256, dur=0.8):
    t = np.arange(0, dur, 1 / fs)
    def one(rare):
        noise = 6.0 * np.random.randn(t.size)
        if rare:
            noise += 5.0 * np.exp(-0.5 * ((t - 0.32) / 0.05) ** 2)
        return noise
    rare = np.stack([one(True) for _ in range(n_rare)])
    freq = np.stack([one(False) for _ in range(n_freq)])
    return t, rare, freq


print("L0 ready")


---

## Session L1 — Inspect a trace and a spectrum

**Aim.** See line noise, a rhythm, and why a spectrum without a window note is incomplete.

**Tasks**

1. Simulate 8 channels for 8 seconds.  
2. Plot 2 seconds of channel 0.  
3. Plot a power spectrum and mark 50 Hz.  
4. Write three sentences: what is neural-like, what is artefact-like, what you cannot know from this figure.


In [ ]:
t, X, fs = simulate_eeg()
fig, axes = plt.subplots(2, 1, figsize=(7.6, 4.6))
axes[0].plot(t[t < 2], X[0, t < 2], color="steelblue", linewidth=0.8)
axes[0].set_title("Channel 0 (2 s excerpt)")
axes[0].set_xlabel("time (s)")
axes[0].set_ylabel("a.u.")
freqs = np.fft.rfftfreq(X.shape[1], 1 / fs)
psd = np.abs(np.fft.rfft(X[0] * np.hanning(X.shape[1]))) ** 2
axes[1].plot(freqs[(freqs > 1) & (freqs < 70)], psd[(freqs > 1) & (freqs < 70)], color="sienna")
axes[1].axvline(50, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Spectrum of channel 0")
axes[1].set_xlabel("Hz")
plt.tight_layout()
plt.show()
print("Optional: replace simulate_eeg() with one channel from a public EDF and repeat.")


**L1 write-up (submit).** Montage you would choose if this were scalp EEG; whether a 50 Hz notch is justified; one way this simulation is kinder than a cafeteria recording.

**Optional public data.** PhysioNet EEGMMI or any MOABB EEG dataset. Load *one* channel, one minute. Do not train a decoder in L1.


---

## Session L2 — A P300-style offline pipeline

**Aim.** Time-lock, average, and feel why repetitions exist (Chapter 5).

**Tasks**

1. Simulate rare and frequent epochs.  
2. Plot single trials faint and means bold.  
3. Build a two-feature decoder: mean amplitude in 0.28–0.38 s.  
4. Train on the first two-thirds of each class *in time order*. Test on the last third. Report accuracy *and* balanced accuracy.


In [ ]:
t, rare, freq = simulate_p300()
mask = (t >= 0.28) & (t <= 0.38)


def amp(epochs):
    return epochs[:, mask].mean(axis=1)


fig, axes = plt.subplots(1, 2, figsize=(7.8, 3.2), sharey=True)
axes[0].plot(t, freq.T, color="steelblue", alpha=0.08, linewidth=0.6)
axes[0].plot(t, freq.mean(0), color="navy", linewidth=2)
axes[0].set_title("Frequent")
axes[1].plot(t, rare.T, color="darkorange", alpha=0.25, linewidth=0.6)
axes[1].plot(t, rare.mean(0), color="sienna", linewidth=2)
axes[1].set_title("Rare")
for ax in axes:
    ax.set_xlabel("s from flash")
axes[0].set_ylabel("a.u.")
plt.tight_layout()
plt.show()

zr, zf = amp(rare), amp(freq)
# time-order split within class
cut_r, cut_f = int(0.67 * zr.size), int(0.67 * zf.size)
thr = 0.5 * (zr[:cut_r].mean() + zf[:cut_f].mean())
y_test = np.hstack([np.ones(zr.size - cut_r), np.zeros(zf.size - cut_f)])
s_test = np.hstack([zr[cut_r:], zf[cut_f:]])
y_hat = (s_test > thr).astype(float)
acc = np.mean(y_hat == y_test)


def bal_acc(y, yh):
    return 0.5 * (np.mean(yh[y == 1] == 1) + np.mean(yh[y == 0] == 0))


print(f"threshold={thr:.3f}  acc={acc:.3f}  balanced={bal_acc(y_test, y_hat):.3f}")
print(f"chance accuracy if always frequent: {zf.size / (zr.size + zf.size):.3f}")
print("A live speller would spend additional seconds repeating flashes (Ch. 5, 9).")


**L2 write-up.** Why balanced accuracy and raw accuracy disagreed, or why they did not. Where this offline test still is *not* a loop (Chapter 4). One sentence on a language-model prior you did *not* add.


---

## Session L3 — Motor-imagery features with a leakage-safe split

**Aim.** Regularised linear decoder, causal blocks, and a shuffled split that you will refuse (Chapters 6–7).


In [ ]:
Z, y, block = simulate_imagery_blocks()


def rlda_fit(Z, y, shrink=0.3):
    S = np.cov(Z, rowvar=False)
    S = (1 - shrink) * S + shrink * np.eye(S.shape[0]) * np.trace(S) / S.shape[0]
    m0, m1 = Z[y == 0].mean(0), Z[y == 1].mean(0)
    w = np.linalg.solve(S, m1 - m0)
    b = -0.5 * w @ (m0 + m1)
    return w, b


def predict(Z, w, b):
    return (Z @ w + b > 0).astype(int)


train, test = block <= 4, block >= 5
w, b = rlda_fit(Z[train], y[train])
acc_causal = np.mean(predict(Z[test], w, b) == y[test])

idx = np.random.permutation(len(y))
cut = int(0.6 * len(y))
w2, b2 = rlda_fit(Z[idx[:cut]], y[idx[:cut]])
acc_leak = np.mean(predict(Z[idx[cut:]], w2, b2) == y[idx[cut:]])

print(f"causal later-blocks accuracy = {acc_causal:.3f}")
print(f"shuffled-trial accuracy      = {acc_leak:.3f}")
print("chance = 0.500; baseline = this rLDA itself")
print("Submit the causal number. Mention the shuffled number only to bury it.")


**L3 write-up.** Reporting bundle items you can already fill. One artefact you *cannot* audit because the simulator did not include EMG — and why that omission would halt a neural claim on a forehead band (Chapter 6).

**Optional public data.** MOABB `BNCI2014_001` or PhysioNet imagery. Extract trial-wise log-band-power yourself or with MNE; keep the *block* split. Do not use the library’s default shuffle CV as your headline.


---

## Session L4 — Design document and ethics checklist

**Aim.** Write before you record. No new decoder.

Copy the following into a markdown cell or a separate `design_L4.md` and fill every line for a hypothetical study you might actually run next term (healthy volunteers, public data, or a clinic partnership — pick one and stay there).


### L4 template

1. Intended-use sentence (Definition 12.2).  
2. User and setting. Wearer-directed or other-directed (11.1 / 11.2)?  
3. Purpose theme; active / reactive / passive.  
4. Paradigm family (Ch. 5) and why the user can produce the generator.  
5. Modality and resolution–risk point (Ch. 3).  
6. S0–S9 in one sentence each.  
7. Latency budget: window, filter delay, compute, UI.  
8. What $\theta$ is; calibration minutes; update rule.  
9. S6 prior or policy, including fade or language model.  
10. Split; chance; baseline; artefact audit.  
11. User-facing metric and day table (Ch. 9).  
12. If improvement is claimed: sham (Ch. 10). Otherwise write “not claimed.”  
13. Data: what is stored, who can join identity to traces, deletion date (12.3).  
14. Owner in year five if anything is implanted or subscribed (3.4).  
15. Access-complete list (12.4): who cannot use this.  
16. The sentence you refuse to put on a poster.

**L4 assessment.** A complete sheet with one refused sentence beats an incomplete sheet with a promised accuracy.


---

## Honour rules for group work

- You may share plots. You may not share a test-block label file.  
- Synthetic cells may be compared. Public-data work must name the dataset and licence.  
- No recording of classmates “for fun” without the local ethics path. The appendix does not grant one.

---

## Mapping onto a teaching week

| Session | Duration | Chapter anchors |
|---|---|---|
| L1 | 90 min | 2, 3, 6 |
| L2 | 90 min | 5, 6, 7, 9 |
| L3 | 90 min | 6, 7 |
| L4 | 90 min | 4, 11, 12 |

---

*End of Appendix L.*  
*End of the laboratory course.*
